-- U2M 
in VS Code - 
run cmds - 
1. databricks -v
2. databricks auth login --host https://........
3. databricks auth profiles
4. databricks workspace list /


In [0]:
-- 2
databricks bundle init

In [0]:
--3
-- 
databricks bundle validate
databricks bundle deploy -t dev

![](/Workspace/Users/sakshijain7945@gmail.com/DE_Assignment/day-9/task3.png)

#INTERMEDIATE

###4.
- Add another target in databricks.yml with a different workspace host and configure the Job to run as a service principal.
- targets: 
  dev: 
    mode: development 
    default: true 
    workspace: host: https://..................
  prod: 
    workspace: host: https://..........
    run_as: service_principal_name: <service_principal_id>

#5
- Created a service principal in the workspace under Identity and access
- Generated a client secret for it
- Added a prod target in databricks.yml pointing to a different host, with run_as set to the service principal
- Set up a CLI profile authenticated via M2M using the service principal instead of personal login
- Ran databricks bundle deploy with that profile to prod, deployment completed successfully
- Confirmed the job was created under the service principal using databricks jobs list

###6
Write a GitHub Actions workflow that runs databricks bundle validate on every pull request, without
deploying anything.

- GitHub automatically recognizes YAML files inside .github/workflows/ as Actions workflows.
- create validate.yml file in it and write the commands here.

developer raise a pull request --> then github action starts --> it downloads GitHub repository onto the temporary GitHub runner --> Install CLI on runner --> then bundle validate , which may be pass/fail

#ADVANCED

###7
- Extend the GitHub Actions workflow to deploy to prod on merge to main using OIDC authentication
(no stored secrets), including the correct permissions: id-token: write block.

DATABRICKS_AUTH_TYPE: github-oidc
tells the Databricks CLI to use GitHub's OIDC identity token. Databricks then exchanges that short-lived token for Databricks authentication. This avoids storing a long-lived Databricks client secret.

```
name: Validate and Deploy
on: pull_request: branches: [main] push: branches: [main]
permissions: 
    id-token: write 
    contents: read

jobs: validate: runs-on: ubuntu-latest 
steps: 
- uses: actions/checkout@v4 
- uses: databricks/setup-cli@main 
- name: Validate bundle run: databricks bundle validate 
env: DATABRICKS_HOST: 
{{ secrets.DATABRICKS_CLIENT_ID }} DATABRICKS_CLIENT_SECRET: ${{ secrets.DATABRICKS_CLIENT_SECRET }}

deploy: if: github.ref == 'refs/heads/main' && github.event_name == 'push' needs: validate runs-on: ubuntu-latest permissions: id-token: write contents: read steps: - uses: actions/checkout@v4 - uses: databricks/setup-cli@main - name: Deploy to prod via OIDC run: databricks bundle deploy -t prod env: DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }} ARM_USE_OIDC: true.```

#8
- Check the job — run it, confirm it's actually broken -databricks bundle run <job_name> -t prod
- Find the last working commit -git log --oneline
- Get the old working files back -git checkout -- databricks.yml resources/
- Redeploy the old version -databricks bundle deploy -t prod
- Confirm it works again -databricks bundle run <job_name> -t prod
- Revert the bad commit in git (so history stays clean) -git revert -git push origin main

#9

Local Setup (one-time) Install Databricks CLI on your machine Authenticate: databricks auth login --host --profile (U2M — browser login) Confirm: databricks current-user me
Edit the Bundle
Our pipeline is code, structured as a Databricks Asset Bundle:

databricks.yml — main config (bundle name, targets, variables) 
resources/*.yml — job/pipeline definitions src/ — notebooks and scripts

Edit these locally using VS Code (not Notepad — YAML is indentation-sensitive).

Test Locally Before Pushing :
- databricks bundle validate --profile 
- databricks bundle deploy --profile -t dev 
- databricks bundle run <job_name> --profile -t dev
Dev deploys are prefixed with your username ([dev yourname] job_name), so testing never collides with teammates.

Open a Pull Request
Pushing your branch and opening a PR against main triggers CI:
databricks bundle validate
This only checks the bundle is valid — nothing gets deployed at this stage.

Merge to Main → Automatic Prod Deploy
On merge, CI/CD runs:
databricks bundle deploy -t prod

using OIDC authentication — no stored secrets. GitHub proves its identity with a short-lived token, and the job runs under a service principal, not a personal account, keeping prod stable and independent of any one person.

If Something Breaks databricks bundle run <job_name> -t prod # confirm it's broken git checkout -- databricks.yml resources/ databricks bundle deploy -t prod # rollback git revert git push origin main